# Idempotency
Use a request key so retries do not repeat a side effect.


In [ ]:
# Reusing one request key returns the original result instead of charging twice.
processed: dict[str, str] = {}

def charge(key: str) -> str:
    if key not in processed:
        processed[key] = "payment-1"
    return processed[key]

print(charge("request-1"), charge("request-1"))


## Polished version
Serialize requests sharing one key and cache the successful result.


In [ ]:
# A per-key lock also protects against two identical requests arriving together.
import asyncio
from collections.abc import Awaitable, Callable
from typing import TypeVar

T = TypeVar("T")

class MemoryIdempotencyStore:
    def __init__(self) -> None:
        self.results: dict[str, object] = {}
        self.locks: dict[str, asyncio.Lock] = {}

    async def run_once(self, key: str, operation: Callable[[], Awaitable[T]]) -> T:
        # Different keys can run concurrently; only duplicates share this lock.
        lock = self.locks.setdefault(key, asyncio.Lock())
        async with lock:
            # Store only a successfully completed operation result.
            if key not in self.results:
                self.results[key] = await operation()
            return self.results[key]  # type: ignore[return-value]

class PaymentService:
    def __init__(self, store: MemoryIdempotencyStore) -> None:
        self.store = store
        self.charges = 0
    async def charge(self, key: str) -> str:
        async def create_charge() -> str:
            self.charges += 1
            await asyncio.sleep(0.01)
            return f"payment-{self.charges}"
        return await self.store.run_once(key, create_charge)

payments = PaymentService(MemoryIdempotencyStore())
results = await asyncio.gather(payments.charge("request-1"), payments.charge("request-1"))
print(results, payments.charges)
